In [8]:
import torch 
import torch.nn as nn
from model import TransformerBlock, LayerNorm, GELU, FeedForward, MultiHeadAttention

In [9]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12, 
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [10]:
class GPTModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embedding = nn.Embedding(config["vocab_size"], config["emb_dim"])
        self.position_embedding = nn.Embedding(config["context_length"], config["emb_dim"])
        self.dropout = nn.Dropout(config["drop_rate"])
        self.transformer_block = nn.Sequential(
            *[TransformerBlock(config) for _ in range(config["n_layers"])]
        )
        self.final_norm = LayerNorm(config["emb_dim"])
        self.output_head = nn.Linear(
            config["emb_dim"], config["vocab_size"], bias=False
        )

    def forward(self, x):
        batch_size, seq_length = x.shape
        token_emb = self.token_embedding(x)
        pos_emb = self.position_embedding(torch.arange(seq_length, device=x.device))
        x = token_emb + pos_emb
        x = self.dropout(x)
        x = self.transformer_block(x)
        x = self.final_norm(x)
        logits = self.output_head(x)
        return logits

In [12]:
torch.manual_seed(123)
gpt_model = GPTModel(GPT_CONFIG_124M)
sample_input = torch.randint(0, GPT_CONFIG_124M["vocab_size"], (2, 10))
output_logits = gpt_model(sample_input)
print("Output logits shape:", output_logits.shape)
print("Input shape:", sample_input.shape)

Output logits shape: torch.Size([2, 10, 50257])
Input shape: torch.Size([2, 10])


In [13]:
total_params = sum(p.numel() for p in gpt_model.parameters())
print(f"Total parameters in GPT-2 124M model: {total_params:,}")

Total parameters in GPT-2 124M model: 155,922,432
